# 05 — Population catchment, rail market and market potential; purchasing power as a catchment characteristic

The demand-side quantities of the revision (`claude_code_prompts_market_potential.md`, P1–P7; register CON-1,
CON-5, T25, T33, VAL-2, VAL-7, VAL-8, REP-9, STA-6), on the frozen 02 / 03 / 04 outputs. Code:
`traintranv2/nb05_catchment.py`, `nb05_market_potential.py`.

| § | step | output (`output/kosice/explore/`) |
|---|---|---|
| 1 | the municipal catchment table (inbound, headline runs), with its reconciliation | `catchment.csv` |
| 2 | three mass levels M1 / M2 / M3; the Slovak cascade and the four-country headline | `catchment.csv`, `cascade.csv` |
| 3 | market potential: contour (primary) and exponential by half-life | `market_potential_contour.csv`, `market_potential_exp.csv` |
| 4 | uncertainty: nominal rate intervals; the DEGURBA corners | `degurba_corners.csv` |
| 5 | purchasing power as a characteristic of the catchment | `pps_profile.csv`, `pps_bands.csv` |
| 5b | car ownership, likewise (CON-11) | `car_profile.csv` |
| 6 | the value index: data check (not built) | — |
| 7 | the coverage-curve figure | `figures/coverage_curve_by_country.png` |

**Standing constraints.**
1. FL551 is a **non-probability quota** survey: every interval derived from it is **nominal**. FL499 (probability)
   is not used here.
2. The FL551 rate is an **upper bound** on the rail-market share. A cross-instrument test against FL499 `dx1` shows
   the quota panel reporting fewer non-travellers than a probability sample in all five countries. Every
   FL551-derived figure carries `rate_is_upper_bound = True`.
3. **Poland** is measured on 32 early-summer days (both Polish feeds, 03b); its figures are labelled so.

**Units.** One row per municipality (LAU). The city of Košice (22 *mestské časti*) is the destination and appears
nowhere; Bratislava's 17 districts are one unit. Population: end-2025, **aged 15+** (both surveys sample 15+); the
0–14 band is carried only to report the share left out. Ukraine has 2024 totals without age bands and is not in
FL551: M1 is its total, reported separately; no rail-market figure.

In [1]:
import os, sys

# Run from the project root so every relative path resolves the same from notebooks/ or the root.
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    os.chdir(os.path.dirname(_here))
sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

Working directory: E:\notes\api_zssk\project


In [2]:
import datetime as dt
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import traintranv2  # noqa: F401  (PROJ data fix)
from traintranv2.config import default_config
from traintranv2 import nb02_connections as C
from traintranv2 import nb05_catchment as K
from traintranv2 import nb05_market_potential as MP
from traintranv2.nb01a_feeds import write_meta
from traintranv2.nb01b_stations import load_lau_subset

cfg = default_config()
OUT = os.path.join(cfg.out_dir, 'explore')
FIG = os.path.join(OUT, 'figures')
os.makedirs(FIG, exist_ok=True)
L = C.lau_with_city_units(load_lau_subset(cfg), cfg)
lau = pd.read_csv(os.path.join(OUT, 'lau_table.csv'), dtype={'lau_gisco_id': str})
rates = K.load_rates('data/eb')
META = {'source': 'Flash Eurobarometer 551 (GESIS ZA8901 v1.0.0, doi:10.4232/1.14506); traintranv2 02-04 outputs',
        'sampling': K.SAMPLING, 'interval_type': K.INTERVAL, 'rate_is_upper_bound': True,
        'generated_at': dt.datetime.now().isoformat(timespec='seconds')}
display(rates.round(3))

,r_national,r_15_64,r_65plus,ci_low,ci_high,ci_low_15_64,ci_high_15_64,ci_low_65plus,ci_high_65plus,t2_given_t1
isocntry,,,,,,,,,,
SK,0.503,0.507,0.488,0.470,0.535,0.471,0.543,0.414,0.562,0.594
CZ,0.594,0.646,0.433,0.563,0.625,0.611,0.681,0.370,0.495,0.686
HU,0.447,0.466,0.387,0.415,0.480,0.429,0.504,0.322,0.451,0.477
PL,0.508,0.547,0.382,0.477,0.540,0.511,0.582,0.317,0.447,0.619
AT,0.496,0.519,0.420,0.464,0.528,0.482,0.556,0.357,0.483,0.531


## 1. The catchment table (P1)

One row per LAU reachable **inbound** on the headline runs. `service_level` is exclusive: a municipality with any
direct train is `direct` and never counted in the one-hop layer. Its journey time, distances and frequency are those
of that level. Purchasing power: NUTS3 PPS per inhabitant (Eurostat `nama_10r_3gdp`) for all five EU countries,
one common year (2023, the latest Austria and Poland publish at NUTS3); `pps_level` and `pps_year` per row; Ukraine
`UA_no_PPS`.

In [3]:
cat = K.catchment(cfg, lau, L)
rec = K.check_catchment(cat)
cov_d = pd.read_csv(os.path.join(cfg.out_dir, 'domestic', 'one_transfer_coverage.csv'))
cov_f = pd.read_csv(os.path.join(cfg.out_dir, 'foreign', 'one_transfer_coverage.csv'))
ref = pd.concat([cov_d.assign(country='SK'), cov_f[cov_f.run.isin(['CZ', 'HU', 'AT', 'PL', 'UA'])]], ignore_index=True)
ref = ref[(ref.direction == 'inbound') & (ref.group == 'direct_or_one_transfer')][['country', 'n_lau', 'population']]
chk = rec.groupby('country').agg(n_lau=('n_lau', 'sum'), pop_total=('pop_total', 'sum')).join(ref.set_index('country'), rsuffix='_0203')
print('reconciliation: the catchment table against the 02 / 03 coverage (direct or one transfer, inbound):')
display(chk)
assert (chk['n_lau'] == chk['n_lau_0203']).all(), 'catchment LAUs differ from 02 / 03 coverage'
display(rec)
print(f"0-14 left out of every rate base: {cat['pop_0_14'].sum():,.0f} "
      f"({cat['pop_0_14'].sum() / cat['pop_total'].sum():.1%} of the catchment's population, UA excluded)")

reconciliation: the catchment table against the 02 / 03 coverage (direct or one transfer, inbound):


,n_lau,pop_total,n_lau_0203,population
country,,,,
AT,31,3160473.0,31,3160473.0
CZ,535,6068419.0,535,6068419.0
HU,186,3844552.0,186,3844552.0
PL,21,4193156.0,21,4193156.0
SK,463,2907245.0,463,2907245.0
UA,3,240896.0,3,240896.0


,country,service_level,n_lau,pop_total,pop_15_plus,pop_0_14
0,AT,onehop,31,3160473.0,2721037.0,439436.0
1,CZ,direct,14,2119181.0,1808312.0,310869.0
2,CZ,onehop,521,3949238.0,3371757.0,577481.0
3,HU,direct,10,1865994.0,1638924.0,227070.0
4,HU,onehop,176,1978558.0,1706410.0,272148.0
5,PL,onehop,21,4193156.0,3619164.0,573992.0
6,SK,direct,107,1413379.0,1202685.0,210694.0
7,SK,onehop,356,1493866.0,1278206.0,215660.0
8,UA,direct,3,240896.0,0.0,0.0


0-14 left out of every rate base: 2,827,350 (13.8% of the catchment's population, UA excluded)


## 2. Mass levels (P2)

* **M1** population catchment = population 15+.
* **M2** rail-travelling population = pop 15–64 · r(15–64) + pop 65+ · r(65+), the age-banded FL551 rates; the
  flat-rate version (r national) beside it.
* **M3** connecting-adjusted = M2 on direct rows, M2 · (T2 | T1) on one-hop rows. A traveller reaching Košice
  directly never connects, so the behavioural discount applies to the one-hop layer only; Poland and Austria have
  no direct train, so their whole catchment takes it.

The conversion applies only **beyond 50 km rail distance**: FL551 Q1_1 describes journeys of 50 km and more, so it
says nothing about the immediate hinterland (Prešov is 34 km away). The inner ring is reported at M1, unconverted,
and is excluded from every converted total.

In [4]:
m = K.mass_levels(cat, rates)
K.check_mass_levels(m)
ring = m[m.in_inner_ring]
print(f'inner ring (< 50 km): {len(ring)} LAUs, {ring.M1.sum():,.0f} people aged 15+ (M1 only, unconverted)')
tot = m[~m.in_inner_ring].groupby(['country', 'service_level'])[['M1', 'M2', 'M2_flat', 'M3']].sum(min_count=1)
tot['M2_flat_vs_banded'] = (tot['M2_flat'] / tot['M2'] - 1).round(4)
display(tot.round(0))

# Slovak cascade, beside the legacy figures (register CON-1)
sk_all = L[(L.CNTR_CODE == 'SK') & ~L.GISCO_ID.isin(C.hub_city_laus(cfg))]
dtrip = pd.read_csv(os.path.join(cfg.out_dir, 'domestic', 'daytrip_one_transfer_lau.csv'), dtype={'lau_gisco_id': str})
same_day = set(dtrip['lau_gisco_id'])
msk = m[m.country == 'SK']
conv_sd = msk[msk.lau_code.isin(same_day)]
LEG = cfg.legacy_dir is not None          # legacy figures exist for Košice only
lg = lambda v: v if LEG else np.nan
cascade = pd.DataFrame([
    {'step': f'resident population (all Slovak LAUs except {cfg.hub_label})', 'base': 'total', 'new': sk_all.POP_2025.sum(), 'legacy': lg(5_195_773)},
    {'step': 'with a rail connection (direct or one transfer)', 'base': 'total', 'new': msk.pop_total.sum(), 'legacy': lg(2_920_024)},
    {'step': 'able to make a same-day return', 'base': 'total', 'new': msk[msk.lau_code.isin(same_day)].pop_total.sum(), 'legacy': lg(1_683_430)},
    {'step': 'rail travellers among them (M2, beyond 50 km)', 'base': '15+', 'new': conv_sd.M2.sum(), 'legacy': lg(846_494)},
    {'step': 'rail travellers among them, flat rate (legacy definition: total population x r_national)', 'base': 'total',
     'new': (msk[msk.lau_code.isin(same_day)].pop_total * rates.loc['SK', 'r_national']).sum(), 'legacy': lg(846_494)},
])
cascade['new_share_of_residents'] = cascade['new'] / cascade.loc[0, 'new']
cascade['legacy_share'] = cascade['legacy'] / cascade.loc[0, 'legacy']
display(cascade.round(3))
four = m[(m.country.isin(['CZ', 'HU', 'PL', 'AT'])) & (m.service_level == 'onehop')]
print(f"four-country one-transfer catchment (CZ, HU, PL, AT; no direct train): total population {four.pop_total.sum():,.0f}, "
      f"15+ {four.M1.sum():,.0f} -> rail market M2 {four.M2.sum():,.0f}"
      + ("  (legacy: 11.25 M -> 5.89 M, total population x flat rate)" if LEG else ""))
if LEG:
    print('why the figures differ: corrected hub (Košice station), Košice as one city, Bratislava one unit, LAU 2024 and 2025 '
          'population, midnight / dedup / same-train fixes, both Polish feeds (CF-23), 15+ base, the 50 km floor')

inner ring (< 50 km): 52 LAUs, 163,412 people aged 15+ (M1 only, unconverted)


M1         M2    M2_flat         M3  \
country service_level                                               
AT      onehop         2721037.0  1356596.0  1350328.0   719799.0   
CZ      direct         1808312.0  1079708.0  1074622.0  1079708.0   
        onehop         3371757.0  1995045.0  2003727.0  1368298.0   
HU      direct         1632342.0   729822.0   729919.0   729822.0   
        onehop         1705276.0   760623.0   762532.0   362848.0   
PL      onehop         3619164.0  1831454.0  1839302.0  1134484.0   
SK      direct         1065465.0   535069.0   535757.0   535069.0   
        onehop         1259730.0   632771.0   633441.0   376135.0   
UA      direct               NaN        NaN        NaN        NaN   

                       M2_flat_vs_banded  
country service_level                     
AT      onehop                      -0.0  
CZ      direct                      -0.0  
        onehop                       0.0  
HU      direct                       0.0  
        onehop                       0.0  
PL      onehop                       0.0  
SK      direct                       0.0  
        onehop                       0.0  
UA      direct                       NaN

,step,base,new,legacy,new_share_of_residents,legacy_share
0,resident population (all Slovak LAUs except Ko...,total,5187121.000,5195773,1.000,1.000
1,with a rail connection (direct or one transfer),total,2907245.000,2920024,0.560,0.562
2,able to make a same-day return,total,2132571.000,1683430,0.411,0.324
3,"rail travellers among them (M2, beyond 50 km)",15+,837000.668,846494,0.161,0.163
4,"rail travellers among them, flat rate (legacy ...",total,1072339.177,846494,0.207,0.163


four-country one-transfer catchment (CZ, HU, PL, AT; no direct train): total population 13,281,425, 15+ 11,418,368 -> rail market M2 5,943,717  (legacy: 11.25 M -> 5.89 M, total population x flat rate)
why the figures differ: corrected hub (Košice station), Košice as one city, Bratislava one unit, LAU 2024 and 2025 population, midnight / dedup / same-train fixes, both Polish feeds (CF-23), 15+ base, the 50 km floor


## 3. Market potential (P3)

**Contour (primary; no free parameter):** Σ M over the LAUs with mean journey time ≤ τ, a curve over τ ∈ {60 … 360}
minutes, per country × service level × mass level. **Exponential:** Σ M · 2^(−t/h), with the half-life h ∈ {60 …
240} minutes as the parameter — a half-life is explicable, a bare decay coefficient is not; the grid over h is the
sensitivity analysis. Reported as **effective population** (population-equivalents at zero journey time), its ratio
to Σ M (what the geography costs), and its share of the national rail market (M2 of the whole country). Cost =
journey time only: a frequency or transfer term would add parameters not calibratable here.

In [5]:
nat = {}
A = L.set_index('GISCO_ID')
for c in rates.index:
    x = A[A.CNTR_CODE == c]
    nat[c] = float((x.POP_15_64_2025 * rates.loc[c, 'r_15_64'] + x.POP_65PLUS_2025 * rates.loc[c, 'r_65plus']).sum())
conv = m      # inner-ring rows carry no M2 / M3, so every converted total leaves them out; M1 keeps them
cont = MP.contour(m)
expo = MP.exponential(m, nat)
pts = cont[(cont.country == 'SK') & (cont.service_level == 'direct') & (cont.mass == 'M1') & cont.tau_min.isin([60, 90])]
print('domestic direct, share of the catchment (15+) within 60 / 90 minutes:',
      dict(zip(pts.tau_min, pts.share_of_catchment.round(3))), '(manuscript: 14.5 % / 30 % of the total population)')
display(cont[(cont.mass == 'M1') & cont.tau_min.notna()].pivot_table(index=['country', 'service_level'], columns='tau_min',
                                                                       values='share_of_catchment').round(3))
display(expo[(expo.mass == 'M2') & (expo.service_level == 'direct_or_onehop')]
        .pivot_table(index='country', columns='half_life_min', values=['effective_population', 'share_national_rail_market']).round(3))

domestic direct, share of the catchment (15+) within 60 / 90 minutes: {60.0: 0.141, 90.0: 0.295} (manuscript: 14.5 % / 30 % of the total population)


tau_min                   60.0   90.0   120.0  180.0  240.0  300.0  360.0
country service_level                                                    
CZ      direct              NaN    NaN    NaN    NaN  0.002  0.064  0.216
        direct_or_onehop    NaN    NaN    NaN    NaN  0.001  0.022  0.082
        onehop              NaN    NaN    NaN    NaN    NaN    NaN  0.011
HU      direct            0.001  0.008  0.082  0.090  1.000  1.000  1.000
        direct_or_onehop  0.001  0.004  0.040  0.051  0.529  0.602  0.693
        onehop              NaN    NaN    NaN  0.014  0.078  0.220  0.399
PL      direct_or_onehop    NaN    NaN    NaN    NaN  0.003  0.003  0.010
        onehop              NaN    NaN    NaN    NaN  0.003  0.003  0.010
SK      direct            0.141  0.295  0.312  0.394  0.538  0.577  1.000
        direct_or_onehop  0.068  0.148  0.162  0.250  0.386  0.473  0.739
        onehop              NaN  0.009  0.020  0.114  0.243  0.376  0.494

effective_population                                      \
half_life_min                  60          90          120         180   
country                                                                  
AT                        4175.093   27306.752   70619.412  184867.694   
CZ                       12497.508   60686.237  146542.859  378348.080   
HU                       84320.060  198547.615  315009.284  512781.793   
PL                         962.940    6080.853   20999.107   85204.358   
SK                      102996.188  192465.903  280077.507  428001.760   

                          share_national_rail_market                       \
half_life_min         240                        60     90     120    180   
country                                                                     
AT             301063.865                      0.001  0.007  0.018  0.047   
CZ             623457.628                      0.002  0.011  0.027  0.069   
HU             661307.324                      0.023  0.055  0.087  0.141   
PL             178817.924                      0.000  0.000  0.001  0.005   
SK             539435.490                      0.045  0.084  0.122  0.186   

                      
half_life_min    240  
country               
AT             0.077  
CZ             0.113  
HU             0.182  
PL             0.011  
SK             0.235

## 4. Uncertainty (P4)

* **The propensity rates are uncertain** (nominal ±3.1 to ±3.3 pp at country level). One rate applies to every
  municipality of a country, so the errors are perfectly correlated: Var(Σ Nᵢ p̂) = (Σ Nᵢ)² Var(p̂), and the
  relative interval of any total equals that of the rate. Age-banding moves the point estimate, not the width. The
  `low` / `high` columns of the M2 / M3 outputs carry these nominal bounds.
* **The half-life is not uncertain, it is unknown:** a grid, never an interval, and never combined with the rate
  interval into one error bar.
* **DEGURBA composition** (bounded secondary sensitivity): applying a country rate uniformly assumes the same
  propensity in cities, towns and rural areas. Corners only: city / town / rural ratios 1.2 / 1 / 0.8 and
  0.8 / 1 / 1.2, against the uniform 1 / 1 / 1. The FL551 rate is a national average, so the ratios are normalised
  over each country's whole 15+ population mix: the national rail market stays the same, and a catchment more urban
  than its country gains under the city-high corner.

In [6]:
x = conv.groupby('country')[['M2', 'M2_low', 'M2_high']].sum(min_count=1)
x['rel_halfwidth'] = ((x['M2_high'] - x['M2_low']) / 2 / x['M2']).round(4)
x['rate_rel_halfwidth'] = ((rates['ci_high'] - rates['ci_low']) / 2 / rates['r_national']).reindex(x.index).round(4)
display(x)
corners = MP.degurba_corners(conv, L)
display(corners.round({'value': 0, 'base': 0}))

,M2,M2_low,M2_high,rel_halfwidth,rate_rel_halfwidth
country,,,,,
AT,1.356596e+06,1.241340e+06,1.471852e+06,0.0850,0.0647
CZ,3.074753e+06,2.857698e+06,3.291807e+06,0.0706,0.0525
HU,1.490445e+06,1.343434e+06,1.637456e+06,0.0986,0.0724
PL,1.831454e+06,1.676417e+06,1.986490e+06,0.0847,0.0619
SK,1.167840e+06,1.062564e+06,1.273116e+06,0.0901,0.0646
UA,NaN,NaN,NaN,NaN,NaN


,country,ratio_city,ratio_town,ratio_rural,mass,value,base
0,AT,1.0,1.0,1.0,M2,1356596.0,1356596.0
1,CZ,1.0,1.0,1.0,M2,3074753.0,3074753.0
2,HU,1.0,1.0,1.0,M2,1490445.0,1490445.0
3,PL,1.0,1.0,1.0,M2,1831454.0,1831454.0
4,SK,1.0,1.0,1.0,M2,1167840.0,1167840.0
5,UA,1.0,1.0,1.0,M2,0.0,NaN
6,AT,1.2,1.0,0.8,M2,1600203.0,1356596.0
7,CZ,1.2,1.0,0.8,M2,3377719.0,3074753.0
8,HU,1.2,1.0,0.8,M2,1654639.0,1490445.0
9,PL,1.2,1.0,0.8,M2,2159961.0,1831454.0


## 5. Purchasing power: a characteristic of the catchment, not a weight (P5; CON-5, T25)

**The index is not multiplied by PPS**, for two reasons:
1. **It would double-count income.** The FL551 rate is *observed* behaviour, not income-standardised: part of the gap
   between Czechia's 59.4 % and Hungary's 44.7 % is an income difference between those populations. Multiplying by
   PPS on top counts income twice. FL551 carries no income variable (occupation `d5a` is the nearest), so the rate
   cannot be decomposed to remove it.
2. **PPS is regional, not municipal.** One NUTS3 value applies to every municipality of the region, so as a weight it
   adds no variation within a region, and spreading it over municipalities does not reduce its error (VAL-8, point 4).
   Ukraine has no PPS at all.

**Aim 3 is narrowed accordingly:** the catchment is weighted by population and degree of urbanisation; regional
purchasing power is reported as a characteristic of the catchment. Below: catchment and rail market by national PPS
quartile (population-weighted over each country's LAUs), the population-weighted mean PPS of the catchment against
the country, direct against one-hop (does a transfer reach different economic territory?). Ukraine is never pooled.

In [7]:
pps = K.pps_table(cfg)
prof, bands = MP.pps_profile(conv, L, pps)
display(prof.round(3))
display(bands.pivot_table(index=['country', 'service_level'], columns='pps_band', values='M1', aggfunc='sum').round(0))
# Bratislava and the Košice region (legacy: 479,389 inhabitants, a third of the directly connected population;
# Košice region 5th nationally, 44th among the sampled EU regions)
# the other city unit in the catchment (Bratislava when the hub is Košice, and the reverse)
from traintranv2.config import DESTINATIONS
other = next(u for u in cfg.city_units if u != cfg.hub_city)
other_nuts3 = next(d['hub_nuts3'] for d in DESTINATIONS.values() if d['hub_city'] == other)
_raw = load_lau_subset(cfg)
n_districts = int(((_raw.CNTR_CODE == 'SK') & _raw.LAU_NAME_NATIONAL.astype(str).str.startswith(cfg.city_units[other])).sum())
ba = m[m.lau_code == f'SK_CITY_{other}']
d_sk = m[(m.country == 'SK') & (m.service_level == 'direct')]
print(f"{other} ({n_districts} districts, one unit): {ba.pop_total.sum():,.0f} inhabitants, {ba.service_level.iloc[0] if len(ba) else '-'}; "
      f"{ba.pop_total.sum() / d_sk.pop_total.sum():.1%} of the directly connected population; "
      f"PPS {pps.set_index('nuts3_id').loc[other_nuts3, 'pps_value']:.0f}; "
      f"journey {(ba.journey_time_min.iloc[0] if len(ba) else float('nan')):.0f} min"
      + (" (legacy 479,389, about a third)" if LEG else ""))
p = pps.dropna(subset=['pps_value']).sort_values('pps_value', ascending=False).reset_index(drop=True)
rk_n = p[p.country == 'SK'].reset_index(drop=True)
H3 = cfg.hub_nuts3
print(f"{cfg.hub_label} region {H3}: PPS {pps.set_index('nuts3_id').loc[H3, 'pps_value']:.0f}; "
      f"rank {rk_n.index[rk_n.nuts3_id == H3][0] + 1} of {len(rk_n)} in Slovakia, "
      f"{p.index[p.nuts3_id == H3][0] + 1} of {len(p)} NUTS3 regions of the five EU countries "
      + (f"(legacy: 5th nationally, 44th; year {int(pps.pps_year.iloc[0])})" if LEG else f"(year {int(pps.pps_year.iloc[0])})"))

,country,service_level,pps_level,n_lau,pps_catchment_popweighted,pps_country_popweighted,catchment_vs_country
0,AT,onehop,NUTS3,31,52924.385,46439.335,1.140
1,AT,direct_or_onehop,NUTS3,31,52924.385,46439.335,1.140
2,CZ,direct,NUTS3,14,59437.949,35119.374,1.692
3,CZ,onehop,NUTS3,521,29056.855,35119.374,0.827
4,CZ,direct_or_onehop,NUTS3,535,39662.601,35119.374,1.129
5,HU,direct,NUTS3,10,58551.381,29096.326,2.012
6,HU,onehop,NUTS3,176,22047.522,29096.326,0.758
7,HU,direct_or_onehop,NUTS3,186,39931.252,29096.326,1.372
8,PL,onehop,NUTS3,21,56260.707,29283.934,1.921
9,PL,direct_or_onehop,NUTS3,21,56260.707,29283.934,1.921


pps_band                         Q1         Q2         Q3         Q4
country service_level                                               
AT      direct_or_onehop    79347.0   288678.0  2016895.0   336117.0
        onehop              79347.0   288678.0  2016895.0   336117.0
CZ      direct             490645.0    91746.0    28260.0  1197661.0
        direct_or_onehop  1724851.0  1012618.0   761478.0  1681122.0
        onehop            1234206.0   920872.0   733218.0   483461.0
HU      direct             148208.0    16917.0        0.0  1473799.0
        direct_or_onehop   765216.0   425397.0   517213.0  1637508.0
        onehop             617008.0   408480.0   517213.0   163709.0
PL      direct_or_onehop   130895.0    30667.0   614566.0  2843036.0
        onehop             130895.0    30667.0   614566.0  2843036.0
SK      direct             271634.0   299828.0   142604.0   488619.0
        direct_or_onehop   653299.0   499560.0   583237.0   744795.0
        onehop             381665.0   199732.0   440633.0   256176.0

Bratislava (17 districts, one unit): 480,902 inhabitants, direct; 34.0% of the directly connected population; PPS 58000; journey 333 min (legacy 479,389, about a third)
Košice region SK042: PPS 24000; rank 6 of 8 in Slovakia, 88 of 150 NUTS3 regions of the five EU countries (legacy: 5th nationally, 44th; year 2023)


## 5b. Car ownership: also a characteristic (CON-11)

Among the demand determinants Reviewer 1 lists, car ownership is the one with harmonised regional data: passenger
cars per 1,000 inhabitants, Eurostat `tran_r_vehst`, NUTS2, 2023 (fetched once into `data/eu/eurostat/` with a
manifest of URL, retrieval time and SHA-256). Like PPS it describes the catchment and is not a weight: it is
regional, and its effect on rail use is what FL551's observed rate already contains. Ukraine has no figure.

In [8]:
import json
cars = pd.read_csv('data/eu/eurostat/tran_r_vehst_car_p_thab.csv')
man = json.load(open('data/eu/eurostat/tran_r_vehst_car_p_thab.manifest.json', encoding='utf-8'))
print(f"tran_r_vehst retrieved {man['retrieved_utc']}, sha256 {man['sha256_json'][:12]}, {man['regions_requested']} NUTS2 regions")
carp = MP.car_profile(conv, L, cars)
display(carp.round(3))

tran_r_vehst retrieved 2026-11-16T10:28:49+00:00, sha256 f981e960f5fb, 46 NUTS2 regions


,country,service_level,level,year,n_lau,cars_per_1000_catchment,cars_per_1000_country,catchment_vs_country
0,AT,onehop,NUTS2,2023,31,448.653,565.541,0.793
1,AT,direct_or_onehop,NUTS2,2023,31,448.653,565.541,0.793
2,CZ,direct,NUTS2,2023,14,679.661,597.967,1.137
3,CZ,onehop,NUTS2,2023,521,577.040,597.967,0.965
4,CZ,direct_or_onehop,NUTS2,2023,535,612.864,597.967,1.025
5,HU,direct,NUTS2,2023,10,419.373,435.305,0.963
6,HU,onehop,NUTS2,2023,176,430.843,435.305,0.990
7,HU,direct_or_onehop,NUTS2,2023,186,425.224,435.305,0.977
8,PL,onehop,NUTS2,2023,21,613.895,594.956,1.032
9,PL,direct_or_onehop,NUTS2,2023,21,613.895,594.956,1.032


## 6. The value index (P6): not built

Before any PPS-based spending proxy, the data check: does a direct measure exist? **Yes.** ŠÚ SR (`cr3804mr`, notebook
00_d1) publishes Košice visitors **and nights by country of origin**, so the average length of stay by origin is
measured directly. A direct measure beats a proxy; the value index is therefore not constructed.

In [9]:
if cfg.visitors_file and os.path.exists(cfg.visitors_file):
    v = pd.read_csv(cfg.visitors_file)
    v = v[v.period == 'YEAR'].pivot_table(index='origin_code', columns='indic_code', values='value')
    v = v.rename(columns={'U_CR_0005': 'visitors', 'U_CR_0008': 'nights', 'U_CR_0011': 'avg_nights'})
    display(v.loc[['VISIT_DOM', 'CZ', 'HU', 'PL', 'AT', 'UA', 'VISIT_FOR', 'VISIT_TOTAL']])
else:
    print(f'no visitors-by-origin file for {cfg.hub_label}: section not computed')

indic_code,visitors,nights,avg_nights
origin_code,,,
VISIT_DOM,144674.0,264191.0,1.8
CZ,15006.0,28546.0,1.9
HU,8128.0,13032.0,1.6
PL,10401.0,16996.0,1.6
AT,4035.0,5853.0,1.5
UA,18228.0,45920.0,2.5
VISIT_FOR,98948.0,211515.0,2.1
VISIT_TOTAL,243622.0,475706.0,2.0


## 7. The coverage-curve figure

Share of each country's catchment (15+) within a mean journey time τ, one line per origin country; direct and direct
or one-hop as separate series.

In [10]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for k, s in enumerate(['direct', 'direct_or_onehop']):
    x = cont[(cont.mass == 'M1') & (cont.service_level == s) & cont.tau_min.notna()]
    for c, g in x.groupby('country'):
        g = g.sort_values('tau_min')
        if g['total'].iloc[0] and np.isfinite(g['total'].iloc[0]):
            ax[k].plot(g.tau_min, g.share_of_catchment, marker='o', label=c)
    ax[k].set_title('direct' if s == 'direct' else 'direct or one transfer')
    ax[k].set_xlabel(f'mean journey time to {cfg.hub_label}, minutes')
    ax[k].grid(alpha=0.3)
ax[0].set_ylabel('share of the catchment (15+)')
ax[1].legend(title='origin', frameon=False)
fig.suptitle(f'Catchment within a journey time of {cfg.hub_label}, by origin country (inbound; PL: 32 summer days)')
fig.tight_layout()
fig.savefig(os.path.join(FIG, 'coverage_curve_by_country.png'), dpi=300)
plt.close(fig)
print('[wrote]', os.path.join(FIG, 'coverage_curve_by_country.png'))

[wrote] output\kosice\explore\figures\coverage_curve_by_country.png


## 8. Methods text and outputs

**For the manuscript.** The market-potential index is a **joint supply-and-demand quantity**: municipalities with no
feasible itinerary contribute zero, so it embeds the connectivity result and is not an independent measure of demand.
The survey conversion applies beyond 50 km rail distance only; the inner ring is reported unconverted. The decay
half-life is swept over a grid, not given an interval, because it is unknown rather than uncertain. Regional
purchasing power characterises the catchment rather than weighting it, for the two reasons in §5. The FL551 rate is a
nominal, non-probability estimate and an upper bound on the rail-market share. Poland is observed on 32 early-summer
days.

In [11]:
for f, k, frame in [('catchment.csv', 'mp_catchment', m), ('cascade.csv', 'mp_cascade', cascade),
                    ('market_potential_contour.csv', 'mp_contour', cont), ('market_potential_exp.csv', 'mp_exp', expo),
                    ('degurba_corners.csv', 'mp_degurba_corners', corners), ('pps_profile.csv', 'mp_pps_profile', prof),
                    ('pps_bands.csv', 'mp_pps_bands', bands), ('car_profile.csv', 'mp_car_profile', carp)]:
    out = frame.assign(**META)
    p = os.path.join(OUT, f)
    out.to_csv(p, index=False, encoding='utf-8')
    write_meta(p, k, out, {'Inputs': 'explore/lau_table.csv, data/eb/fl551_*_national.csv, NUTS subset PPS 2023'})
    print(f'[wrote] {p} ({len(out):,} rows) + .meta.md')

[wrote] output\kosice\explore\catchment.csv (1,239 rows) + .meta.md
[wrote] output\kosice\explore\cascade.csv (5 rows) + .meta.md
[wrote] output\kosice\explore\market_potential_contour.csv (360 rows) + .meta.md
[wrote] output\kosice\explore\market_potential_exp.csv (225 rows) + .meta.md
[wrote] output\kosice\explore\degurba_corners.csv (18 rows) + .meta.md
[wrote] output\kosice\explore\pps_profile.csv (13 rows) + .meta.md
[wrote] output\kosice\explore\pps_bands.csv (52 rows) + .meta.md
[wrote] output\kosice\explore\car_profile.csv (13 rows) + .meta.md
